# Notebook 2: Spread Construction

Builds the physically-grounded **partial product spread** using yield fractions
derived from the Aspen HYSYS atmospheric CDU simulation.

## Methodology note — why this is a partial spread, not a full refinery margin

The HYSYS simulation produces a full product slate from the WTI Light crude assay:

| Product | Yield % | Liquid daily futures available? |
|---------|---------|----------------------------------|
| Naphtha  | 3.4%  | Yes — RBOB (`RB=F`) |
| Kerosene | 11.7% | No |
| Diesel   | 12.2% | Yes — Heating Oil (`HO=F`) |
| AGO      | 12.6% | No |
| Residue  | 57.3% | No |

Kerosene, AGO, and residue have no equivalent liquid, exchange-traded daily futures
series. Rather than invent a discount-to-WTI assumption to price them synthetically —
which would introduce an unverified number and undermine the real-data premise of this
project — **this strategy deliberately prices only the naphtha and diesel revenue
streams**, which together represent 15.65% of the barrel.

To keep the economics internally consistent, the crude cost charged in the spread
formula is scaled proportionally to this priced fraction — not the full barrel price.
This avoids the inconsistency of subtracting full crude cost while only crediting
a fraction of the output.

```python
PRICED_YIELD_FRACTION = yield_naphtha + yield_diesel   # = 0.1565

spread = (yield_naphtha * RBOB * 42)
       + (yield_diesel  * HO   * 42)
       - (PRICED_YIELD_FRACTION * WTI)
       - utility_cost_per_bbl
```

**Benchmarked against the generic 3:2:1 crack spread** in Notebook 4, under identical
trading mechanics, to test whether the HYSYS-derived weighting improves risk-adjusted
performance rather than merely being physically motivated.

---

### Data integrity guard

An earlier version of Notebook 1 assigned column names positionally, which silently
swapped the RBOB and Heating Oil series (yfinance returns columns alphabetically).
Assertions on plausible price levels are included below so that class of error cannot
propagate into the spread undetected.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from signal_generator import (
    YIELD_NAPHTHA, YIELD_KEROSENE, YIELD_DIESEL, YIELD_AGO, YIELD_RESIDUE,
    UTILITY_COST_PER_BBL, PRICED_YIELD_FRACTION,
    compute_hysys_margin, compute_generic_321_spread
)

In [ ]:
# ============================================================
# HYSYS-DERIVED INPUTS (imported from src/signal_generator.py
# — single source of truth, kept consistent across all notebooks)
# ============================================================

print('Full HYSYS product slate (basis: 9,336 kg/h crude feed):')
print(f'  Naphtha  (PRICED -> RBOB):   {YIELD_NAPHTHA:.4f}  ({YIELD_NAPHTHA*100:.2f}%)')
print(f'  Kerosene (excluded):         {YIELD_KEROSENE:.4f}  ({YIELD_KEROSENE*100:.2f}%)')
print(f'  Diesel   (PRICED -> HO):     {YIELD_DIESEL:.4f}  ({YIELD_DIESEL*100:.2f}%)')
print(f'  AGO      (excluded):         {YIELD_AGO:.4f}  ({YIELD_AGO*100:.2f}%)')
print(f'  Residue  (excluded):         {YIELD_RESIDUE:.4f}  ({YIELD_RESIDUE*100:.2f}%)')
print(f'\nPriced fraction of barrel:    {PRICED_YIELD_FRACTION:.4f}  ({PRICED_YIELD_FRACTION*100:.2f}%)')
print(f'Utility cost per barrel:      ${UTILITY_COST_PER_BBL:.3f}/bbl')

In [ ]:
# Load price data
df = pd.read_csv('../data/raw_prices.csv', index_col=0, parse_dates=True)
print(f'Loaded {len(df)} trading days')
print(f'Date range: {df.index[0].date()} to {df.index[-1].date()}')
df.head()

In [ ]:
# ============================================================
# DATA INTEGRITY GUARD
# Crude quotes in $/bbl and is an order of magnitude larger than
# the products, which quote in $/gal. Heating oil normally trades
# slightly above RBOB over a long sample. Both checks fail loudly
# if the product columns are ever swapped upstream.
# ============================================================

print('Mean levels:')
print(f"  WTI      ${df['WTI'].mean():7.2f}/bbl")
print(f"  RBOB     ${df['RBOB'].mean():7.3f}/gal")
print(f"  HeatOil  ${df['HeatOil'].mean():7.3f}/gal")

ho_minus_rb = (df['HeatOil'] - df['RBOB']).mean()
print(f"\nHeatOil - RBOB = ${ho_minus_rb:.3f}/gal")
if ho_minus_rb < -0.10:
    print('*** WARNING: strongly negative — product columns may be swapped. ***')
else:
    print('Consistent with correctly labelled series.')

assert df['WTI'].mean()     > 20, 'WTI implausibly low - check column mapping in NB1'
assert df['RBOB'].mean()    < 10, 'RBOB implausibly high - check column mapping in NB1'
assert df['HeatOil'].mean() < 10, 'HeatOil implausibly high - check column mapping in NB1'
print('\nIntegrity assertions passed.')

In [ ]:
# Unit conversion: RBOB and HO are quoted $/gallon -> multiply by 42 to get $/bbl
df['RBOB_bbl']    = df['RBOB']    * 42
df['HeatOil_bbl'] = df['HeatOil'] * 42

# ============================================================
# PARTIAL PRODUCT SPREAD - the strategy's tradeable signal
# ============================================================
df['spread_hysys'] = compute_hysys_margin(df['WTI'], df['RBOB'], df['HeatOil'])

# Generic 3:2:1 crack spread - benchmark only
df['crack_321'] = compute_generic_321_spread(df['WTI'], df['RBOB'], df['HeatOil'])

print('Spread series statistics ($/bbl):')
print(df[['spread_hysys', 'crack_321']].describe().round(3))

In [ ]:
# Plot both spreads for comparison
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

axes[0].plot(df.index, df['spread_hysys'], color='#2c3e50', linewidth=0.8,
             label='HYSYS partial product spread')
axes[0].axhline(df['spread_hysys'].mean(), color='#e74c3c', linestyle='--', linewidth=1,
                label=f'Mean: ${df["spread_hysys"].mean():.2f}/bbl')
axes[0].set_ylabel('Spread ($/bbl)')
axes[0].set_title('HYSYS-Weighted Partial Product Spread (naphtha + diesel, crude cost scaled to 15.65%)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(df.index, df['crack_321'], color='#7f8c8d', linewidth=0.8,
             label='Generic 3:2:1 crack spread (benchmark)')
axes[1].axhline(df['crack_321'].mean(), color='#e74c3c', linestyle='--', linewidth=1,
                label=f'Mean: ${df["crack_321"].mean():.2f}/bbl')
axes[1].set_ylabel('Crack Spread ($/bbl)')
axes[1].set_title('Generic 3:2:1 Crack Spread (full-barrel-cost industry convention - benchmark)')
axes[1].set_xlabel('Date')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/spread_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('The HYSYS spread prices only naphtha + diesel (15.65% of the barrel) with crude')
print('cost scaled proportionally. The 3:2:1 spread charges a full WTI barrel against a')
print('2:1 gasoline:heating-oil weighting and is shown purely as the benchmark (NB4).')

In [ ]:
# Save constructed spread data
df.to_csv('../data/spread_data.csv')
print(f'Saved {len(df)} rows to ../data/spread_data.csv')